# TP final IA II - Paso 3 (versión 2): red CNN + RNN con 6 campañas, 4 clases e imágenes cada 10 días

Entrena y evalúa la red **solo con imágenes** (el clima queda para el notebook 04), usando el dataset del notebook 02 versión 2. Mantiene todo lo que funcionó en el 03b (validación cruzada por departamento, baselines, comparación pareada, retoma automática) y agrega:

1. **Dos preguntas de evaluación:**
   - **Por departamento** (5 particiones): ¿funciona en lugares que no vio?
   - **Dejando un año afuera** (6 particiones, una por campaña): se entrena con 5 campañas y se predice la sexta. ¿Funciona en un año que no vio? Es la prueba más realista, porque es como se usaría en la práctica, y recién ahora se puede hacer.
2. **Experimentos nuevos:**
   - **Cada 10 días contra mensual**: con los mismos datos, promediando de a 3 períodos. ¿Le sirve a la RNN una secuencia más larga?
   - **10 bandas contra 6**: ¿aporta el borde rojo?
3. **Clase Otros** (otros cultivos y no agrícola), para que el modelo sirva en un campo real.
4. **Intervalos de confianza para el rinde** por *bootstrap* de departamentos, para saber si las diferencias son reales.
5. **Validación interna más pareja**: en el 03b, en algunas particiones la validación había quedado con 2 departamentos. Ahora se elige la división interna más cercana al 20 %.
6. **Los baselines también se guardan**: en el 03b el Random Forest tardaba 28 minutos en cada corrida.

Se mantienen los experimentos que en la versión 1 respondieron algo importante (LSTM sola, CNN sin RNN, meses disponibles, oclusión). Los que ya quedaron claros (GRU, píxeles permutados, rinde sobre cosechada) siguen disponibles, pero no corren por defecto, para ahorrar GPU.

**Antes de correrlo:**
- GPU **T4** en *Entorno de ejecución*. El dataset ocupa unos 3 GB en memoria: si Colab avisa que se quedó sin RAM, usá *Entorno de ejecución* → *Cambiar tipo de entorno* → **RAM alta**, si está disponible.
- Correr primero con `PRUEBA = True` (600 parches, 2 épocas, unos 20 minutos) y después con `False`.

**Tiempo:** el entrenamiento completo son unos 50 entrenamientos (estimado: 5 a 7 horas en T4). Cada uno se guarda apenas termina y **si Colab se corta, al volver a ejecutar todo sigue donde quedó**, aunque sea otro día. Las listas `CORRER_DEPTO` y `CORRER_ANIO` de la sección 1 están ordenadas por importancia: si querés, corré primero solo los primeros.

In [ ]:
PRUEBA = False  # True: 600 parches y 2 épocas para probar el circuito. False: entrenamiento completo

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/TP_final_IA2'
except ImportError:
    BASE = os.environ.get('TP_BASE', 'TP_final_IA2')

DATASET  = f'{BASE}/dataset_v2'
SALIDA   = f'{BASE}/resultados_03v2_prueba' if PRUEBA else f'{BASE}/resultados_03v2'
CORRIDAS = f'{SALIDA}/corridas'
os.makedirs(CORRIDAS, exist_ok=True)
for f in ['dataset_cultivos.npz', 'dataset_meta.csv']:
    assert os.path.exists(f'{DATASET}/{f}'), f'Falta {DATASET}/{f}: corré el notebook 02 versión 2'
print('Resultados en', SALIDA)

Mounted at /content/drive
Resultados en /content/drive/MyDrive/TP_final_IA2/resultados_03v2


In [ ]:
import time, json, hashlib, inspect, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, regularizers
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, mean_absolute_error
warnings.filterwarnings('ignore', message='.*shuffle=True.*')
tf.get_logger().setLevel('ERROR')

SEED = 42
keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', gpus if gpus else 'NO HAY GPU: activá T4 (en CPU el entrenamiento completo tarda muchísimo)')

# Precisión mixta: en la T4 las convoluciones van casi el doble de rápido en float16. Las salidas quedan en float32.
MIXTA = bool(gpus)
if MIXTA:
    keras.mixed_precision.set_global_policy('mixed_float16')
print('Precisión mixta:', MIXTA)

TensorFlow 2.21.0 | Keras 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Precisión mixta: True


## 1. Parámetros y experimentos

Cada experimento cambia **una sola cosa** respecto de la red principal (CNN + LSTM, cada 10 días, 10 bandas, rinde sobre superficie sembrada).

In [ ]:
CLASES     = ['Maíz', 'Soja 1ra', 'Soja 2da', 'Otros']
CULTIVOS   = [0, 1, 2]                                            # las clases que tienen rinde
BANDAS_S2  = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
CANALES    = BANDAS_S2 + ['NDVI', 'NDRE']                         # NDRE: índice de borde rojo, (B8A - B5) / (B8A + B5)
CANALES_6  = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI']       # lo mismo que veía la versión 1
T_FULL, PATCH = 27, 16
MESES = ['sep', 'oct', 'nov', 'dic', 'ene', 'feb', 'mar', 'abr', 'may']

N_FOLDS      = 5
EPOCAS       = 2 if PRUEBA else 60
PACIENCIA    = 10
BATCH        = 64
LR           = 1e-3
L2           = 1e-4
DROPOUT      = 0.3
FILTROS      = (16, 32, 64)
UNIDADES_RNN = 64
PESO_RINDE   = 0.5
N_PRUEBA     = 600
RF_ARBOLES   = 100 if PRUEBA else 300
VERSION      = 1

BASE_EXP = dict(arq='cnn_lstm', objetivo='rinde_sembrado', periodos=T_FULL, mensual=False, bandas=10, permutado=False)
EXPERIMENTOS = {
    'CNN + LSTM':                     {},
    'LSTM sola':                      dict(arq='lstm_sola'),
    'CNN sin RNN':                    dict(arq='cnn_sin_rnn'),
    'CNN + LSTM, mensual':            dict(mensual=True),
    'CNN + LSTM, 6 bandas':           dict(bandas=6),
    'CNN + LSTM, hasta nov':          dict(periodos=9),
    'CNN + LSTM, hasta ene':          dict(periodos=15),
    'CNN + LSTM, hasta mar':          dict(periodos=21),
    'CNN + GRU':                      dict(arq='cnn_gru'),
    'CNN + LSTM, píxeles permutados': dict(permutado=True),
    'CNN + LSTM, rinde cosechado':    dict(objetivo='rinde_cosechado'),
}
EXPERIMENTOS = {k: {**BASE_EXP, **v} for k, v in EXPERIMENTOS.items()}
PRINCIPAL = 'CNN + LSTM'

# Qué se corre en cada esquema, en orden de importancia (para correr menos, recortá las listas)
CORRER_DEPTO = ['CNN + LSTM', 'LSTM sola', 'CNN sin RNN', 'CNN + LSTM, mensual', 'CNN + LSTM, 6 bandas',
                'CNN + LSTM, hasta nov', 'CNN + LSTM, hasta ene', 'CNN + LSTM, hasta mar']
                # opcionales, ya respondidos en la versión 1: 'CNN + GRU', 'CNN + LSTM, píxeles permutados', 'CNN + LSTM, rinde cosechado'
CORRER_ANIO  = ['CNN + LSTM', 'LSTM sola']
PERIODOS_RF  = [9, 15, 21, 27]       # el Random Forest también se evalúa viendo hasta nov, ene, mar y may
ESQUEMAS = {'depto': 'por departamento', 'anio': 'dejando un año afuera'}

## 2. Carga y verificación

Igual que en el 03b. Las imágenes se pasan a reflectancia (0-1) y se agregan dos índices: **NDVI** (verdor) y **NDRE**, su equivalente con el borde rojo, más sensible a la clorofila cuando el cultivo ya está cerrado. Quedan 12 canales.

In [ ]:
d = np.load(f'{DATASET}/dataset_cultivos.npz')
X_u16, y_cultivo, ids = d['X'], d['y_cultivo'].astype(np.int64), d['id']
meta = pd.read_csv(f'{DATASET}/dataset_meta.csv')
necesarias = {'id', 'campana', 'clase', 'cultivo', 'grupo', 'rinde_sembrado', 'rinde_cosechado'}
assert not (necesarias - set(meta.columns)), f'Faltan columnas: {necesarias - set(meta.columns)}'
assert meta.id.is_unique and len(meta) == len(ids), 'dataset_meta.csv no coincide con dataset_cultivos.npz'
meta = meta.set_index('id').loc[ids].reset_index()
assert (meta.clase.values == y_cultivo).all(), 'Las clases del npz y del csv no coinciden'
assert X_u16.shape[1:] == (T_FULL, PATCH, PATCH, len(BANDAS_S2)), f'Forma inesperada de X: {X_u16.shape}'

if PRUEBA:      # muestra pareja por campaña y clase
    rng = np.random.default_rng(SEED)
    grupos_cc = meta.groupby(['campana', 'clase']).indices
    por = max(1, N_PRUEBA // len(grupos_cc))
    sel = np.sort(np.concatenate([rng.choice(v, size=min(por, len(v)), replace=False) for v in grupos_cc.values()]))
    X_u16, y_cultivo, ids, meta = X_u16[sel], y_cultivo[sel], ids[sel], meta.iloc[sel].reset_index(drop=True)
N = len(meta)

ib = {b: i for i, b in enumerate(BANDAS_S2)}
X_all = np.empty((N, T_FULL, PATCH, PATCH, len(CANALES)), np.float16)
for i in range(0, N, 500):
    x = X_u16[i:i + 500].astype(np.float32) / 10000.0
    ndvi = (x[..., ib['B8']] - x[..., ib['B4']]) / np.maximum(x[..., ib['B8']] + x[..., ib['B4']], 1e-6)
    ndre = (x[..., ib['B8A']] - x[..., ib['B5']]) / np.maximum(x[..., ib['B8A']] + x[..., ib['B5']], 1e-6)
    X_all[i:i + 500] = np.concatenate([x, ndvi[..., None], ndre[..., None]], axis=-1)
del X_u16, d

S1 = np.zeros((N, len(CANALES))); S2 = np.zeros((N, len(CANALES)))
F_RF = np.zeros((N, T_FULL, len(CANALES)), np.float32)     # Random Forest: promedio de cada canal en cada período
for i in range(0, N, 500):
    x = X_all[i:i + 500].astype(np.float32)
    S1[i:i + 500] = x.mean(axis=(1, 2, 3)); S2[i:i + 500] = (x ** 2).mean(axis=(1, 2, 3))
    F_RF[i:i + 500] = x.mean(axis=(2, 3))

campanas = sorted(meta.campana.unique())
print('X:', X_all.shape, X_all.dtype, f'{X_all.nbytes / 1e9:.2f} GB')
print('\nParches por campaña y clase:')
print(meta.groupby(['campana', 'cultivo']).size().unstack().reindex(columns=CLASES))
por_depto = meta.groupby('grupo').size()
print(f'\nDepartamentos: {len(por_depto)} | los 5 más grandes tienen el {100 * por_depto.nlargest(5).sum() / N:.0f} % de los parches')
print('Parches de cultivos con rinde sembrado:', int(meta.rinde_sembrado.notna().sum()), 'de', int((meta.clase < 3).sum()))

X: (18692, 27, 16, 16, 12) float16 3.10 GB

Parches por campaña y clase:
cultivo  Maíz  Soja 1ra  Soja 2da  Otros
campana                                 
2019_20   800       800       800    613
2020_21   800       800       800    800
2021_22   800       800       800    800
2022_23   800       800       800    479
2023_24   800       800       800    800
2024_25   800       800       800    800

Departamentos: 37 | los 5 más grandes tienen el 25 % de los parches
Parches de cultivos con rinde sembrado: 14400 de 14400


## 3. Particiones

- **Por departamento:** `StratifiedGroupKFold` con 5 particiones; cada departamento es test una sola vez. Estrato = clase x campaña.
- **Dejando un año afuera:** 6 particiones, el test de cada una es una campaña entera. Los mismos departamentos aparecen en train con otros años, como pasaría en la práctica, porque de un lugar se conoce su historia.

En las dos, la validación interna (para el EarlyStopping) se separa **por departamento** dentro de train, eligiendo entre 5 divisiones posibles la que deje más cerca del 20 % de los parches.

In [ ]:
estrato = y_cultivo + len(CLASES) * meta.campana.map({c: i for i, c in enumerate(campanas)}).values
grupos = meta.grupo.values

def particion_interna(tr, seed):
    sg = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    a, b = min(sg.split(tr, estrato[tr], grupos[tr]), key=lambda ab: abs(len(ab[1]) / len(tr) - 0.2))
    return np.sort(tr[a]), np.sort(tr[b])

FOLDS = {'depto': [], 'anio': []}
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(sgkf.split(np.zeros(N), estrato, grupos)):
    a, b = particion_interna(tr, SEED + k)
    FOLDS['depto'].append({'train': a, 'val': b, 'test': np.sort(te), 'nombre': f'partición {k}'})
for k, camp in enumerate(campanas):
    te = np.flatnonzero(meta.campana.values == camp)
    tr = np.flatnonzero(meta.campana.values != camp)
    a, b = particion_interna(tr, SEED + 100 + k)
    FOLDS['anio'].append({'train': a, 'val': b, 'test': te, 'nombre': camp})

for esq, folds in FOLDS.items():
    filas = []
    for f in folds:
        assert len(set(f['train']) & set(f['test'])) == 0 and len(set(f['val']) & set(f['test'])) == 0
        assert not (set(grupos[f['train']]) & set(grupos[f['val']])), 'Departamentos repetidos entre train y validación'
        if esq == 'depto':
            assert not (set(grupos[f['train']]) & set(grupos[f['test']])), 'Departamentos repetidos entre train y test'
        filas.append({'test': f['nombre'], 'parches train': len(f['train']), 'parches val': len(f['val']),
                      'parches test': len(f['test']), 'deptos val': len(set(grupos[f['val']])),
                      'deptos test': len(set(grupos[f['test']]))})
    todos = np.sort(np.concatenate([f['test'] for f in folds]))
    assert np.array_equal(todos, np.arange(N)), f'{esq}: cada parche tiene que ser test exactamente una vez'
    print(f'\nEsquema {ESQUEMAS[esq]}:')
    print(pd.DataFrame(filas).to_string(index=False))

# Se guardan para que el notebook 04 use exactamente las mismas particiones
part = pd.DataFrame({'id': ids})
for esq, folds in FOLDS.items():
    part[f'test_{esq}'] = -1
    for k, f in enumerate(folds):
        part.loc[f['test'], f'test_{esq}'] = k
part.to_csv(f'{SALIDA}/particiones_03.csv', index=False)


Esquema por departamento:
       test  parches train  parches val  parches test  deptos val  deptos test
partición 0          11212         3059          4421           5            8
partición 1          11975         3037          3680           5            8
partición 2          11675         2988          4029           7            7
partición 3          12416         3266          3010           7            6
partición 4          12589         2551          3552           5            8

Esquema dejando un año afuera:
   test  parches train  parches val  parches test  deptos val  deptos test
2019_20          12959         2720          3013           6           37
2020_21          12352         3140          3200           7           37
2021_22          12387         3105          3200           6           37
2022_23          12507         3306          2879           8           37
2023_24          12215         3277          3200           7           37
2024_25          

## 4. Predicciones antes de correr

Borrador para revisar y escribir con sus palabras **antes** de pasar a `PRUEBA = False`. Algunas se apoyan en lo que mostró la versión 1.

1. **Cada 10 días contra mensual.** Con 27 pasos, la LSTM tiene más secuencia para aprovechar (por ejemplo, el momento exacto de la cosecha del trigo). Esperamos una mejora chica en cultivo, y que ahora la CNN + LSTM le saque alguna ventaja a la CNN sin RNN.
2. **10 bandas contra 6.** El borde rojo es sensible a la clorofila: esperamos poca diferencia en cultivo y alguna mejora en rinde.
3. **LSTM sola contra CNN + LSTM.** Igual que en la versión 1: los parches siguen siendo puros y ahora son más chicos, así que la CNN no debería aportar.
4. **La clase Otros** va a ser la más difícil, porque mezcla girasol, sorgo, pasturas y montes. Esperamos confusiones con maíz y soja 1ra.
5. **Dejando un año afuera.** La clasificación cae un poco, sobre todo al predecir 2022/23 (la sequía cambia la forma de las curvas). En rinde, la pregunta clave es si el modelo le gana al **promedio histórico del cultivo**: esperamos que sí en los años extremos, porque la sequía se ve en la imagen.
6. **Fechas disponibles.** Como en la versión 1: soja 2da reconocible desde noviembre y maíz contra soja recién hacia marzo.

## 5. Baselines

| Baseline | Qué mide |
|---|---|
| **Media de train** | Predice siempre el rinde medio. |
| **Referencia** | Por departamento: el rinde medio de train del cultivo y la campaña **reales** del parche. Dejando un año afuera, esa campaña no está en train, así que pasa a ser el **promedio histórico del cultivo** en los otros 5 años: lo que diría alguien que solo conoce la historia. |
| **Random Forest** | Promedio de cada canal en cada período (27 x 12 = 324 variables). También con los períodos hasta nov, ene y mar. |

Se calculan una vez por partición y se guardan: la próxima corrida los carga en segundos. **La primera vez tardan bastante (más de una hora con el dataset completo)**, porque el Random Forest corre en el procesador y no en la GPU: el notebook va mostrando cada partición que termina.

In [ ]:
OBJETIVOS = ['rinde_sembrado', 'rinde_cosechado']
PRED, FILAS = {}, []

def nuevo_pred():
    return {'pc': np.full(N, -1), 'pr': np.full(N, np.nan)}

def metricas(idx, pc, pr, objetivo):
    out = {}
    if pc is not None:
        out['accuracy'] = accuracy_score(y_cultivo[idx], pc)
        out['F1 macro'] = f1_score(y_cultivo[idx], pc, average='macro', labels=range(len(CLASES)), zero_division=0)
        out['F1 cultivos'] = f1_score(y_cultivo[idx], pc, average='macro', labels=CULTIVOS, zero_division=0)
    real = meta[objetivo].values[idx]
    ok = np.isfinite(real) & np.isfinite(pr)
    if ok.sum():
        out['MAE rinde (kg/ha)'] = mean_absolute_error(real[ok], pr[ok])
        df = meta.iloc[idx][['grupo', 'cultivo', 'campana']].assign(real=real, pred=pr)[ok]
        g = df.groupby(['grupo', 'cultivo', 'campana']).agg(real=('real', 'first'), pred=('pred', 'mean'))
        out['MAE por depto (kg/ha)'] = mean_absolute_error(g.real, g.pred)
    return out

def media_referencia(idx_train, idx_eval, objetivo):
    tr, ev = meta.iloc[idx_train], meta.iloc[idx_eval]
    m_cc = tr.groupby(['cultivo', 'campana'])[objetivo].mean()
    m_c = tr.groupby('cultivo')[objetivo].mean()
    pr = pd.Series(list(zip(ev.cultivo, ev.campana))).map(m_cc).values.astype(float)
    pr = np.where(np.isnan(pr), ev.cultivo.map(m_c).values.astype(float), pr)     # campaña no vista -> promedio histórico
    return np.where(ev.clase.values >= 3, np.nan, pr)

def baselines_fold(esq, k):
    ruta = f'{CORRIDAS}/{esq}/baselines_fold{k}.json'
    f = FOLDS[esq][k]
    clave = hashlib.md5((str(VERSION) + str(RF_ARBOLES) + str(PRUEBA) + ids[f['train']].tobytes().hex()[:5000] +
                         ids[f['test']].tobytes().hex()[:5000]).encode()).hexdigest()[:12]
    if os.path.exists(ruta):
        res = json.load(open(ruta))
        if res.get('huella') == clave:
            return res
    tr, te = f['train'], f['test']
    res = {'huella': clave, 'idx_test': te.tolist(), 'pred': {}}
    for objetivo in OBJETIVOS:
        yr = meta[objetivo].values
        res['pred'][f'Media de train|{objetivo}'] = {'pr': np.where(meta.clase.values[te] >= 3, np.nan, np.nanmean(yr[tr])).tolist()}
        res['pred'][f'Referencia|{objetivo}'] = {'pr': media_referencia(tr, te, objetivo).tolist()}
    for p in PERIODOS_RF:
        Ftr, Fte = F_RF[tr, :p].reshape(len(tr), -1), F_RF[te, :p].reshape(len(te), -1)
        clf = RandomForestClassifier(n_estimators=RF_ARBOLES, min_samples_leaf=2, max_features='sqrt', n_jobs=-1,
                                     random_state=SEED, class_weight='balanced').fit(Ftr, y_cultivo[tr])
        pc = clf.predict(Fte)
        nombre = 'Random Forest' if p == T_FULL else f'Random Forest, hasta {MESES[(p - 1) // 3]}'
        for objetivo in (OBJETIVOS if p == T_FULL else ['rinde_sembrado']):
            yr = meta[objetivo].values
            ok = np.isfinite(yr[tr])
            reg = RandomForestRegressor(n_estimators=RF_ARBOLES, min_samples_leaf=2, max_features='sqrt', n_jobs=-1,
                                        random_state=SEED).fit(Ftr[ok], yr[tr][ok])
            pr = np.where(meta.clase.values[te] >= 3, np.nan, reg.predict(Fte))
            res['pred'][f'{nombre}|{objetivo}'] = {'pc': pc.tolist(), 'pr': pr.tolist()}
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    json.dump(res, open(ruta, 'w'))
    return res

def registrar(esq, nombre, objetivo, k, idx, pc, pr, extra=None):
    p = PRED.setdefault((esq, nombre, objetivo), nuevo_pred())
    if pc is not None:
        p['pc'][idx] = pc
    p['pr'][idx] = pr
    FILAS.append({'esquema': esq, 'modelo': nombre, 'objetivo': objetivo, 'fold': k,
                  **metricas(idx, pc, pr, objetivo), **(extra or {})})

t0 = time.time()
for esq in FOLDS:
    for k in range(len(FOLDS[esq])):
        t1 = time.time()
        res = baselines_fold(esq, k)
        print(f'  baselines {ESQUEMAS[esq]}, {FOLDS[esq][k]["nombre"]}: {time.time() - t1:.0f} s', flush=True)
        idx = np.array(res['idx_test'])
        for clave, v in res['pred'].items():
            nombre, objetivo = clave.split('|')
            pc = np.array(v['pc']) if 'pc' in v else None
            registrar(esq, nombre, objetivo, k, idx, pc, np.array(v['pr'], dtype=float))
print(f'Baselines listos en {time.time() - t0:.0f} s')
b = pd.DataFrame(FILAS)
print(b[b.objetivo == 'rinde_sembrado'].groupby(['esquema', 'modelo'], sort=False)[['F1 macro', 'MAE por depto (kg/ha)']]
      .mean().round(3).to_string())

  baselines por departamento, partición 0: 1 s
  baselines por departamento, partición 1: 1 s
  baselines por departamento, partición 2: 1 s
  baselines por departamento, partición 3: 1 s
  baselines por departamento, partición 4: 1 s
  baselines dejando un año afuera, 2019_20: 1 s
  baselines dejando un año afuera, 2020_21: 1 s
  baselines dejando un año afuera, 2021_22: 1 s
  baselines dejando un año afuera, 2022_23: 1 s
  baselines dejando un año afuera, 2023_24: 1 s
  baselines dejando un año afuera, 2024_25: 1 s
Baselines listos en 9 s
                                  F1 macro  MAE por depto (kg/ha)
esquema modelo                                                   
depto   Media de train                 NaN               1809.155
        Referencia                     NaN                756.123
        Random Forest, hasta nov     0.798                996.188
        Random Forest, hasta ene     0.903                822.805
        Random Forest, hasta mar     0.932               

## 6. Las redes

Misma estructura que en el 03b, adaptada a parches de 16 x 16:
- **CNN por período** (`TimeDistributed`): dos bloques de dos convoluciones 3 x 3 + *max pooling* (16 → 8 → 4) y una convolución final con *global average pooling*: un vector de 64 valores por período.
- **Parte temporal:** LSTM o GRU de 64 unidades; `cnn_sin_rnn` pone todos los períodos uno al lado del otro en una capa densa; `lstm_sola` no tiene convoluciones y usa el promedio de cada canal del parche.
- **Dos cabezas:** `cultivo` (4 clases) y `rinde` (lineal, rinde estandarizado, pérdida Huber). La clase Otros y los parches sin dato llevan **peso 0** en el rinde.

Las variantes **mensual**, **6 bandas** y **hasta nov/ene/mar** usan exactamente la misma red: lo único que cambia es qué parte de los datos recibe.

In [ ]:
def construir_modelo(arq, T, C):
    reg = regularizers.l2(L2)
    entrada = keras.Input(shape=(T, PATCH, PATCH, C), name='serie')
    if arq == 'lstm_sola':
        x = layers.TimeDistributed(layers.GlobalAveragePooling2D(), name='promedio_por_periodo')(entrada)
    else:
        x = entrada
        for f in FILTROS[:-1]:
            x = layers.TimeDistributed(layers.Conv2D(f, 3, padding='same', activation='relu', kernel_regularizer=reg))(x)
            x = layers.TimeDistributed(layers.Conv2D(f, 3, padding='same', activation='relu', kernel_regularizer=reg))(x)
            x = layers.TimeDistributed(layers.MaxPooling2D())(x)
        x = layers.TimeDistributed(layers.Conv2D(FILTROS[-1], 3, padding='same', activation='relu', kernel_regularizer=reg))(x)
        x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
        x = layers.Dropout(DROPOUT)(x)
    if arq in ('cnn_lstm', 'lstm_sola'):
        x = layers.LSTM(UNIDADES_RNN, name='lstm')(x)
    elif arq == 'cnn_gru':
        x = layers.GRU(UNIDADES_RNN, name='gru')(x)
    elif arq == 'cnn_sin_rnn':
        x = layers.Flatten(name='periodos_juntos')(x)
    else:
        raise ValueError(arq)
    x = layers.Dropout(DROPOUT)(x)
    comun = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    cultivo = layers.Dense(len(CLASES), activation='softmax', name='cultivo', dtype='float32')(comun)
    r = layers.Concatenate()([comun, cultivo])
    r = layers.Dense(32, activation='relu', kernel_regularizer=reg)(r)
    rinde = layers.Dense(1, name='rinde', dtype='float32')(r)
    modelo = keras.Model(entrada, {'cultivo': cultivo, 'rinde': rinde}, name=arq)
    modelo.compile(optimizer=keras.optimizers.Adam(LR),
                   loss={'cultivo': 'sparse_categorical_crossentropy', 'rinde': keras.losses.Huber(delta=1.0)},
                   loss_weights={'cultivo': 1.0, 'rinde': PESO_RINDE},
                   metrics={'cultivo': ['accuracy']}, weighted_metrics={'rinde': ['mae']})
    return modelo

def forma_entrada(cfg):
    T = 9 if cfg['mensual'] else cfg['periodos']
    C = len(CANALES_6) if cfg['bandas'] == 6 else len(CANALES)
    return T, C

for arq in ['cnn_lstm', 'cnn_gru', 'cnn_sin_rnn', 'lstm_sola']:
    print(f'{arq:12s}: {construir_modelo(arq, T_FULL, len(CANALES)).count_params():>7,} parámetros')

cnn_lstm    :  76,133 parámetros
cnn_gru     :  68,069 parámetros
cnn_sin_rnn : 149,605 parámetros
lstm_sola   :  26,373 parámetros


In [ ]:
# Los lotes se arman directamente desde X_all (numpy), sin pasar los 3 GB a TensorFlow:
# convertirlos a un tensor los duplicaba (o triplicaba) en memoria y Colab se quedaba sin RAM.
PERM = np.random.default_rng(SEED).permutation(PATCH * PATCH).astype(np.int32)
np.random.seed(SEED)

def stats_canales(idx):
    mu = S1[idx].mean(axis=0)
    sd = np.sqrt(np.maximum(S2[idx].mean(axis=0) - mu ** 2, 1e-12)) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)

def preparar_rinde(objetivo, idx_train):
    yr = meta[objetivo].values.astype(np.float64)
    if objetivo in meta:
        yr = np.where(meta.clase.values >= 3, np.nan, yr)
    mu, sd = np.nanmean(yr[idx_train]), np.nanstd(yr[idx_train])
    peso = np.isfinite(yr).astype(np.float32)
    yr_std = np.where(np.isfinite(yr), (yr - mu) / sd, 0.0).astype(np.float32)
    return yr, yr_std, peso, mu, sd

def pesos_clase(y):
    n = np.bincount(y, minlength=len(CLASES)).astype(np.float32)
    return (len(y) / (len(CLASES) * np.maximum(n, 1)))[y].astype(np.float32)

def aumentar(x):
    # x: (n, T, H, W, C). Mismo giro y espejado para todos los períodos de cada parche.
    for j in range(len(x)):
        x[j] = np.rot90(x[j], np.random.randint(4), axes=(1, 2))
        if np.random.rand() < 0.5:
            x[j] = x[j][:, :, ::-1]
    return x

def armar_ds(idx, yr_std, peso_r, mu_c, sd_c, entrenar, cfg):
    T, C = forma_entrada(cfg)
    canales = np.array([CANALES.index(c) for c in (CANALES_6 if cfg['bandas'] == 6 else CANALES)])
    mu, sd = mu_c[canales], sd_c[canales]
    w_c = pesos_clase(y_cultivo[idx]) if entrenar else np.ones(len(idx), np.float32)

    def lote(ii):
        x = X_all[ii].astype(np.float32)
        x = (x[..., canales] - mu) / sd
        if cfg['mensual']:
            x = x.reshape(len(ii), 9, 3, PATCH, PATCH, C).mean(axis=2)     # promedio de los 3 períodos de cada mes
        else:
            x = x[:, :T]
        if entrenar:
            x = aumentar(np.ascontiguousarray(x))
        if cfg['permutado']:
            x = x.reshape(len(ii), T, PATCH * PATCH, C)[:, :, PERM].reshape(len(ii), T, PATCH, PATCH, C)
        return np.ascontiguousarray(x, dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((idx.astype(np.int64), y_cultivo[idx].astype(np.int32), yr_std[idx], w_c, peso_r[idx]))
    if entrenar:
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)

    def cargar(i, c, r, wc, wr):
        x = tf.ensure_shape(tf.numpy_function(lote, [i], tf.float32), [None, T, PATCH, PATCH, C])
        return x, {'cultivo': c, 'rinde': r}, {'cultivo': wc, 'rinde': wr}

    return ds.batch(BATCH).map(cargar, num_parallel_calls=2).prefetch(tf.data.AUTOTUNE)

### Entrenamiento con retoma automática

Igual que en el 03b: cada entrenamiento (esquema, experimento, partición) se guarda en `corridas/` y, si ya existe con la misma huella, se carga. Para la red principal se guarda también el modelo y la **oclusión por período**: se tapa un período por vez (se reemplaza por el promedio de train) y se mide cuánto cae el F1.

In [ ]:
def _md5(b):
    return hashlib.md5(b if isinstance(b, bytes) else b.encode()).hexdigest()

def slug(nombre):
    return re.sub(r'[^a-z0-9]+', '_', nombre.lower().replace('í', 'i')).strip('_')

def huella(esq, nombre, k):
    cfg = EXPERIMENTOS[nombre]
    try:
        codigo = _md5(''.join(inspect.getsource(fn) for fn in
                              (construir_modelo, forma_entrada, armar_ds, aumentar, pesos_clase, stats_canales, preparar_rinde)))
    except (OSError, TypeError):
        codigo = 'sin-fuente'
    f = FOLDS[esq][k]
    h = dict(version=VERSION, esq=esq, cfg=cfg, epocas=EPOCAS, paciencia=PACIENCIA, batch=BATCH, lr=LR, l2=L2,
             dropout=DROPOUT, filtros=FILTROS, unidades=UNIDADES_RNN, peso_rinde=PESO_RINDE, seed=SEED, prueba=PRUEBA,
             codigo=codigo, **{n: _md5(ids[f[n]].tobytes()) for n in ('train', 'val', 'test')})
    return _md5(json.dumps(h, sort_keys=True))[:12]

def oclusion(modelo, idx, mu_c, sd_c, cfg):
    # por tandas de 512 parches, para no ocupar memoria de más
    T, C = forma_entrada(cfg)
    pred_base, pred_t = [], [[] for _ in range(T)]
    for s in range(0, len(idx), 512):
        x = (X_all[idx[s:s + 512], :T].astype(np.float32) - mu_c) / sd_c
        pred_base.append(modelo.predict(x, batch_size=256, verbose=0)['cultivo'].argmax(1))
        for t in range(T):
            xo = x.copy(); xo[:, t] = 0.0          # el período t pasa a ser "un período promedio de train"
            pred_t[t].append(modelo.predict(xo, batch_size=256, verbose=0)['cultivo'].argmax(1))
    f1 = lambda pc: f1_score(y_cultivo[idx], pc, average='macro', labels=range(len(CLASES)), zero_division=0)
    base = f1(np.concatenate(pred_base))
    return {'f1_base': base, 'caida_f1': [base - f1(np.concatenate(p)) for p in pred_t]}

class Avance(keras.callbacks.Callback):
    """Imprime una línea por época, para ver que el entrenamiento avanza."""
    def __init__(self, etiqueta):
        super().__init__(); self.etiqueta = etiqueta
    def on_train_begin(self, logs=None):
        self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        print(f'    {self.etiqueta} · época {epoch + 1}: val_loss {logs.get("val_loss", float("nan")):.3f} '
              f'({time.time() - self.t0:.0f} s)', flush=True)

def entrenar_corrida(esq, nombre, k):
    cfg = EXPERIMENTOS[nombre]
    h = huella(esq, nombre, k)
    ruta = f'{CORRIDAS}/{esq}/{slug(nombre)}_fold{k}.json'
    if os.path.exists(ruta):
        res = json.load(open(ruta))
        if res.get('huella') == h:
            res['cargada'] = True
            return res
    f = FOLDS[esq][k]
    mu_c, sd_c = stats_canales(f['train'])
    yr, yr_std, peso_r, mu, sd = preparar_rinde(cfg['objetivo'], f['train'])
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED + k)
    T, C = forma_entrada(cfg)
    modelo = construir_modelo(cfg['arq'], T, C)
    cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5)]
    t0 = time.time()
    hist = modelo.fit(armar_ds(f['train'], yr_std, peso_r, mu_c, sd_c, True, cfg),
                      validation_data=armar_ds(f['val'], yr_std, peso_r, mu_c, sd_c, False, cfg),
                      epochs=EPOCAS, callbacks=cb + [Avance(f'{nombre}, {FOLDS[esq][k]["nombre"]}')], verbose=0).history
    segundos = time.time() - t0
    p = modelo.predict(armar_ds(f['test'], yr_std, peso_r, mu_c, sd_c, False, cfg), verbose=0)
    rinde = p['rinde'].ravel().astype(np.float64) * sd + mu
    rinde = np.where(meta.clase.values[f['test']] >= 3, np.nan, rinde)
    res = {'huella': h, 'idx_test': f['test'].tolist(), 'prob': np.round(p['cultivo'].astype(np.float64), 5).tolist(),
           'rinde': [None if np.isnan(v) else float(v) for v in rinde], 'epocas': len(hist['loss']),
           'mejor_epoca': int(np.argmin(hist['val_loss'])) + 1, 'segundos': segundos, 'parametros': modelo.count_params(),
           'historial': {m: [float(v) for v in vals] for m, vals in hist.items()}}
    if nombre == PRINCIPAL:
        canales = [CANALES.index(c) for c in CANALES]
        res['oclusion'] = oclusion(modelo, f['test'], mu_c[canales], sd_c[canales], cfg)
        modelo.save(f'{CORRIDAS}/{esq}/modelo_{slug(nombre)}_fold{k}.keras')
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    json.dump(res, open(ruta, 'w'))
    res['cargada'] = False
    return res

## 7. Correr los experimentos

Una línea por entrenamiento: esquema, experimento, partición, épocas, F1 de test y tiempo. *(guardada)* = cargada de Drive. Si Colab se corta, volvé a ejecutar todo.

In [ ]:
faltan_vars = [v for v in ('CORRER_DEPTO', 'CORRER_ANIO', 'FOLDS', 'X_all', 'PRED') if v not in globals()]
if faltan_vars:
    raise RuntimeError('La sesión de Colab se reinició (se perdió lo que estaba en memoria). '
                       'Usá Entorno de ejecución -> Ejecutar todo: lo que ya terminó (baselines y entrenamientos) se carga de Drive.')
RES = {}
tareas = [('depto', n, k) for n in CORRER_DEPTO for k in range(len(FOLDS['depto']))] + \
         [('anio', n, k) for n in CORRER_ANIO for k in range(len(FOLDS['anio']))]
t_ini, entrenadas, seg = time.time(), 0, 0.0
for j, (esq, nombre, k) in enumerate(tareas, 1):
    r = entrenar_corrida(esq, nombre, k)
    RES[(esq, nombre, k)] = r
    idx = np.array(r['idx_test'])
    f1k = f1_score(y_cultivo[idx], np.argmax(r['prob'], 1), average='macro', labels=range(len(CLASES)), zero_division=0)
    if not r['cargada']:
        entrenadas += 1; seg += r['segundos']
    falta = f' | faltan ~{(len(tareas) - j) * seg / entrenadas / 60:.0f} min' if entrenadas and j < len(tareas) else ''
    estado = '(guardada)' if r['cargada'] else f'{r["segundos"] / 60:.1f} min'
    print(f'[{j:2d}/{len(tareas)}] {esq:5s} {nombre:28s} {FOLDS[esq][k]["nombre"]:12s} {r["epocas"]:3d} ép. '
          f'(mejor {r["mejor_epoca"]:3d}) | F1 {f1k:.3f} | {estado}{falta}')
print(f'\nListo en {(time.time() - t_ini) / 60:.1f} min')

for (esq, nombre, k), r in RES.items():
    objetivo = EXPERIMENTOS[nombre]['objetivo']
    idx = np.array(r['idx_test'])
    pr = np.array([np.nan if v is None else v for v in r['rinde']], dtype=float)
    registrar(esq, nombre, objetivo, k, idx, np.argmax(r['prob'], 1), pr,
              {'épocas': r['mejor_epoca'], 'minutos': r['segundos'] / 60, 'parámetros': r['parametros']})
RESULT = pd.DataFrame(FILAS)

    CNN + LSTM, partición 0 · época 1: val_loss 0.457 (45 s)
    CNN + LSTM, partición 0 · época 2: val_loss 0.456 (78 s)
    CNN + LSTM, partición 0 · época 3: val_loss 0.351 (116 s)
    CNN + LSTM, partición 0 · época 4: val_loss 0.286 (140 s)
    CNN + LSTM, partición 0 · época 5: val_loss 0.275 (181 s)
    CNN + LSTM, partición 0 · época 6: val_loss 0.289 (220 s)
    CNN + LSTM, partición 0 · época 7: val_loss 0.276 (242 s)
    CNN + LSTM, partición 0 · época 8: val_loss 0.279 (264 s)
    CNN + LSTM, partición 0 · época 9: val_loss 0.280 (288 s)
    CNN + LSTM, partición 0 · época 10: val_loss 0.242 (313 s)
    CNN + LSTM, partición 0 · época 11: val_loss 0.246 (334 s)
    CNN + LSTM, partición 0 · época 12: val_loss 0.252 (359 s)
    CNN + LSTM, partición 0 · época 13: val_loss 0.228 (397 s)
    CNN + LSTM, partición 0 · época 14: val_loss 0.231 (441 s)
    CNN + LSTM, partición 0 · época 15: val_loss 0.232 (482 s)
    CNN + LSTM, partición 0 · época 16: val_loss 0.251 (504 s)
   

KeyboardInterrupt: 